# Using Kaggle noteboks as a model server

![](https://cdn-images-1.medium.com/max/880/1*c0pioKlhVSBeX7ya6iPHsQ.jpeg)

## Prerequisites
- [Kaggle account](https://www.kaggle.com/) to run a [Kaggle (code) notebook](https://www.kaggle.com/code).
- Sign up to [ngrok](https://ngrok.com/) and copy your [authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

## Setting up the server

### At the Kaggle notebook instance
1. Start the notebook session.
  - This works with GPU or CPU, make sure to use a model optimized for the chosen hardware.
2. Set up your environment.
  - Download and install [Ollama](https://github.com/ollama/ollama?tab=readme-ov-file#linux)
  - Install ngrok.
  - Load the dependencies.
3. Set the ngrok authtoken and server port.
4. Start `Ollama` server.
5. Set up the `ngrok` credentials.
6. Open the `ngrok` tunnel.
7. Copy the public URL, it will look something like this: `https://ae63-35-230-113-242.ngrok-free.app`.

After following the steps above, you will be able to query the model hosted by Kaggle through your terminal, a notebook, or any other source.

## Prompting the model

### Local terminal
1. On your terminal, update your `OLLAMA_HOST` env variable with `export OLLAMA_HOST={OLLAMA_HOST_URL}` where `OLLAMA_HOST_URL` should be replaced by the address created at the previous step.
2. Load a model onto the server with `ollama run {MODEL_ID}` where `MODEL_ID` could be any [model supported by Ollama](https://ollama.com/search) like `gemma3n`.
3. After running `ollama run {MODEL_ID}`, you will be able to query the model at the terminal.

### Jupyter notebook
The easiest way to query the ollama server would be with the [ollama-python](https://github.com/ollama/ollama-python), but you can also use something like the `requests` library as you would query any other API.

#### I also wrote a blog post about this idea, check it out at [this link](https://medium.com/data-science-collective/create-a-remote-llm-server-using-kaggle-notebooks-and-ollama-acb299ead1e5)

If you want to watch a video walkthrough of this code, take a look at this video that I recorded

# GPU information

In [ ]:
!nvidia-smi

# Setup

In [ ]:
!sudo apt-get install zstd

In [ ]:
# Download and install ollama to the system
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!pip install -qq pyngrok ollama

# Dependencies

In [ ]:
import IPython
import subprocess
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from typing import Any, Dict, List, Optional
import subprocess, os, time

env = os.environ.copy()
# env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["OLLAMA_CONTEXT_LENGTH"]="100000"
env["OLLAMA_KEEP_ALIVE"]="-1"
# env["OLLAMA_NUM_PARALLEL"]="2"
env["OLLAMA_FLASH_ATTENTION"]="1"
# env["OLLAMA_KV_CACHE_TYPE"]="q8_0"


# Auxiliary functions

In [ ]:
def start_ollama_server(env: dict = None) -> None:
    """Starts the Ollama server."""
    subprocess.Popen(['ollama', 'serve'], env=env)
    print("Ollama server started.")


def check_ollama_port(port: str) -> None:
    """Check if Ollama server is running at the specified port."""
    try:
        subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'], check=True, capture_output=True, text=True)
        if any(f":{port} (LISTEN)" in line for line in subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'], capture_output=True, text=True).stdout.splitlines()):
            print(f"Ollama is listening on port {port}")
        else:
            print(f"Ollama does not appear to be listening on port {port}.")
    except subprocess.CalledProcessError as e:
         print(f"Error checking Ollama port: {e}")


def setup_ngrok_tunnel(port: str, secret_manager: UserSecretsClient) -> ngrok.NgrokTunnel:
    """Sets up an ngrok tunnel.

    Args:
        port: The port to tunnel.

    Returns:
        The ngrok tunnel object.

    Raises:
        RuntimeError: If the ngrok authtoken is not set.
    """
    ngrok_auth_token = secret_manager.get_secret("NGROK_TOKEN")
    if not ngrok_auth_token:
        raise RuntimeError("NGROK_AUTHTOKEN is not set.")

    ngrok.set_auth_token(ngrok_auth_token)
    tunnel = ngrok.connect(port, host_header=f'localhost:{port}')
    print(f"ngrok tunnel created: {tunnel.public_url}")
    return tunnel

# Parameters

In [ ]:
NGROK_PORT = '11434'

# Start the `Ollama` server

In [ ]:
start_ollama_server(env=env)

check_ollama_port(NGROK_PORT)

# Setup `ngrok` tunnel

In [ ]:
user_secrets = UserSecretsClient()

ngrok_tunnel = setup_ngrok_tunnel(NGROK_PORT, user_secrets)

# Query the server

In [ ]:
MODEL_ID = 'qwen3.5:9b'

## From a terminal

On your terminal, update your `OLLAMA_HOST` env variable with `export OLLAMA_HOST={OLLAMA_HOST_URL}` where `OLLAMA_HOST_URL` should be replaced by the address created at the previous step.

Copy the command below:


In [ ]:
print(f'export OLLAMA_HOST={ngrok_tunnel.public_url}')

Load a model onto the server with `ollama run {MODEL_ID}` where `MODEL_ID` could be any [model supported by Ollama](https://ollama.com/search) like `gemma3n`.

After running `ollama run {MODEL_ID}` you will be able to query the model at the terminal.


Copy the command below:

In [ ]:
print(f'ollama run {MODEL_ID}')

## From a notebook

### Using `ollama-python` lib

In [ ]:
import ollama


In [ ]:
def query_ollama_with_client(client: ollama.Client, prompt: str, model_id: str) -> None:
    """Queries the Ollama server using the `ollama-python` client library.

    Args:
        prompt: The prompt to send to the model.
        model_id: The ID of the model to use.
    """
    try:
        messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ]

        response = client.chat(
            model=model_id,
            
            messages=messages
        )
        print("Response from ollama client:")
        print(response['message']['content'])
    except Exception as e:
        print(f"Error querying Ollama with client: {e}")

In [ ]:
# Setup an Ollama client with the provided host
client = ollama.Client(host=ngrok_tunnel.public_url)

In [ ]:
# Load the model at the client
client.pull(model=MODEL_ID)

#### Text prompt

In [ ]:
query_ollama_with_client(
    client,
    PROMPT,
    MODEL_ID,
)

#### Multilingual prompt

In [ ]:
query_ollama_with_client(
    client,
    "sebaik apa kamu dalam coding?",
    MODEL_ID,
)